In [6]:
import json
import time
import paho.mqtt.client as mqtt
import logging

def sender(data_list):
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger()

    # Настройки MQTT
    BROKER = "158.160.241.56"
    # BROKER="mqtt.rootsapp.ru"
    # BROKER=""
    PORT = 1883
    # PORT = 8084
    DEVICE_SERIAL = "00000022"
    USERNAME = DEVICE_SERIAL
    PASSWORD = "GMSL00000022"

    MQTT_TOPIC_DATA = f"mqtt/devices/{DEVICE_SERIAL}/data"

    # Callback
    def on_connect(client, userdata, flags, reason_code, properties):
        if reason_code == 0:
            logger.info("Connected to MQTT Broker")
            pass
        else:
            logger.error(f"Failed to connect, reason_code={reason_code}")

    def on_publish(client, userdata, mid, reason_code, properties):
        logger.info(f"Message {mid} published with reason code {reason_code}")

    # Создаём клиент и подключаемся
    client = mqtt.Client(
        client_id=DEVICE_SERIAL, callback_api_version=mqtt.CallbackAPIVersion.VERSION2, 
    )
    # if USERNAME:
    client.username_pw_set(USERNAME, PASSWORD)

    client.on_connect = on_connect
    client.on_publish = on_publish

    client.connect(BROKER, PORT)
    client.loop_start()

    # Публикуем каждую запись по очереди

    for entry in data_list:
        payload = json.dumps(entry)
        client.publish(MQTT_TOPIC_DATA, payload=entry, qos=1)
        logger.info(f"Published: {payload}")
        time.sleep(0.1)  # пауза между публикациями

    client.loop_stop()
    client.disconnect()
    logger.info("Disconnected from MQTT")

sender(["fffffff"])


INFO:root:Published: "fffffff"
INFO:root:Connected to MQTT Broker
INFO:root:Message 1 published with reason code Success
INFO:root:Disconnected from MQTT


In [ ]:
import requests

def send_packet(device_id: str, binary_data: bytes, url: str = "http://test.rootsapp.ru/gpackets") -> int:
    """
    Отправляет сырые бинарные данные на указанный URL с заголовками:
    - Content-Type: application/octet-stream
    - X-Device-Id: <device_id>

    Возвращает HTTP статус-код ответа.
    """
    headers = {
        "Content-Type": "application/octet-stream",
        "X-Device-Id": device_id
    }

    try:
        response = requests.post(url, headers=headers, data=binary_data)
        status = response.status_code

        if status == 202:
            print("Пакет принят, в очереди")
        elif status == 200:
            print("Дубль: пакет уже был")
        elif status == 400:
            print("Ошибка 400: пустой body или отсутствует X-Device-Id")
        elif status == 415:
            print("Ошибка 415: неверный Content-Type")
        else:
            print(f"Неожиданный статус: {status}")

        return status

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return None

# Пример использования:
if __name__ == "__main__":
    # Ваши бинарные данные (например, четыре байта)
    data = b'\x01\x02\x03\x04\x05\x01\x02\x03\x04\x05\x01\x02\x03\x04\x05'
    # daata=[0x01,0x02,0x03,0x04]
    # ID устройства
    device = "00000099"

    # Отправка
    send_packet(device, data)